In [ ]:
### Read camera centers
import json
import numpy as np
import matplotlib.pyplot as plt
import os

from scene.colmap_loader import read_extrinsics_binary, qvec2rotmat

data_type = 'stump'
img_path = f'data/360_v2/{data_type}/sparse/0/images.bin'

cam_exts = read_extrinsics_binary(img_path)
cam_centers = []
cam_rots = []
cam_names = []
for idx, extr in cam_exts.items():
    cam_names.append(os.path.basename(extr.name).split(".")[0])
    rot = qvec2rotmat(extr.qvec)
    trans = np.array(extr.tvec)
    cam_centers.append(-rot.T @ trans)
    cam_rots.append(rot.T)

cam_centers = np.array(cam_centers)
cam_rots = np.array(cam_rots)
print(f'Cam centers: {cam_centers.shape}, Cam rots: {cam_rots.shape}')

### import from json
# with open('logs/atomgs/bicycle_default/cameras.json', 'r') as fs:
#     cam_dict = json.load(fs)
#     cam_jcenters = np.array([cam['position'] for cam in cam_dict])
#     cam_jrots = np.array([cam['rotation'] for cam in cam_dict])
# print(f'Cam jcenters: {cam_jcenters.shape}, Cam jrots: {cam_jrots.shape}')

In [ ]:
u, s, vh = np.linalg.svd(cam_centers-cam_centers.mean(0))
### Plot camera centers
fig = plt.figure(figsize=(5,5), constrained_layout=True)
ax = fig.add_subplot(projection='3d')
ax.scatter(cam_centers[:, 0], cam_centers[:, 1], zs=cam_centers[:, 2], color='g', alpha=0.2)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

ax.plot([0, vh[0][0]], [0, vh[0][1]], zs=[0, vh[0][2]], color='b')
ax.plot([0, vh[1][0]], [0, vh[1][1]], zs=[0, vh[1][2]], color='g')
ax.plot([0, vh[2][0]], [0, vh[2][1]], zs=[0, vh[2][2]], color='r')

cam_dist = cam_centers @ vh[2]
chosen = cam_dist > np.percentile(cam_dist, 80)
sel_centers = cam_centers[chosen]

ax.scatter(sel_centers[:, 0], sel_centers[:, 1], zs=sel_centers[:, 2], color='m')

In [ ]:
### PCA splits
save_path = f'data/360_v2/{data_type}/split_pca3_20top.json'
chosen = cam_dist > np.percentile(cam_dist, 80)
train_set = [cam_names[c] for c in np.flatnonzero(chosen)]
test_set = [cam_names[c] for c in np.flatnonzero(np.logical_not(chosen))]
with open(save_path, 'w+') as fs:
    json.dump({'train': train_set, 'test': test_set}, fs)
print(f'train: {len(train_set)}', f'test: {len(test_set)}')

In [ ]:
### Sort splits
save_path = 'data/360_v2/bicycle/split_sort_20step.json'
interval = 5
invert = True
sorted_names = sorted(cam_names)
train_set = [name for idx, name in enumerate(sorted_names) if idx % interval != 0]
test_set = [name for idx, name in enumerate(sorted_names) if idx % interval == 0]
if invert:
    train_set, test_set = test_set, train_set
with open(save_path, 'w+') as fs:
    json.dump({'train': train_set, 'test': test_set}, fs)
print(len(train_set), len(test_set))

In [ ]:
### Make video of images in train_set and test_set
import json
import PIL.Image
from subprocess import Popen, PIPE
import os

json_path = f'data/360_v2/{data_type}/split_pca3_20top.json'
video_path = json_path.split('.')[0] + '.mp4'
with open(json_path, 'r') as fs:
    split_dict = json.load(fs)
    train_set = split_dict['train']
    test_set = split_dict['test']

# p = Popen(['ffmpeg', '-y', '-f', 'image2pipe', '-vcodec', 'mjpeg', '-r', '24', '-i', '-', '-vcodec', 'mpeg4', '-qscale', '5', '-r', '24', video_path], stdin=PIPE)
p = Popen(['ffmpeg', '-y', '-f', 'image2pipe', '-framerate', '2', '-i', '-', '-c:v', 'mpeg4', '-vf', 'format=yuv420p', '-r', '25', '-movflags', '+faststart', video_path], stdin=PIPE)
for img_name in train_set:
    img_path = os.path.join(os.path.dirname(json_path), 'images', f'{img_name}.JPG')
    with open(img_path, 'rb') as fs:
        img = PIL.Image.open(fs)
        img.save(p.stdin, 'JPEG')
p.stdin.close()
p.wait()
print(len(train_set))

In [ ]:
### Evaluate and render

model_path = '???'
camera_path = '???'
camera_names = '???' ### indicies on camera_path

def render_images(model_path, camera_path, camera_names):
    images = None
    return images

def save_images(images, dir, img_name=None, video_name=None):
    if img_name:
        pass
    elif video_name:
        pass
    return

def eval_images(source, target):
    return dict()

In [ ]:
### Select gaussians for generative training in source and target model
### 1. Restrict source gaussians to those seen by partial view cameras
### 2. Select voxels of a fixed (metric approx) size around each source gaussian
### 3. Restrict target gaussians to those inside each selected source voxel
### 4. (optional) Remove voxels that are not confident (<50% views in target cameras) 
### to train only on good quality samples
### Note: 4 boosts generalization in the expense of source scene reconstruction
### Inference: apply the trained method to only voxels that have low confidence in source views


### Count cameras per gaussian

### Find the 3D window

def window_gaussians(gaussians, window):
    pass